# JiT-S2-VMamba — SSC-static (Mamba-3 bias-only CONTROL)

Control arm of the SSC ladder: ONLY the ones-init static biases on the SSM maps, `B′ = B + b₀`, `C′ = C + c₀` per CrossScan direction (Mamba-3, arXiv 2603.15569, Table 10) — **no condition path through SSC** (no `W_B`/`W_C`; the class/timestep reach the model only via adaLN-Zero). This is exactly what the `bc` arm equals at step 0, trained to convergence without ever opening the condition projections: it prices the pure static-bias effect so `bc` − `static` isolates the DiM-2 conditioning effect. Ladder: baseline → **static** → bc → abc. Params: +1.5K.

`ssc: "static"` is NOT on main yet — section 3 applies an idempotent 5-edit patch to the cloned `src/models/vmamba.py` (backup kept, syntax-checked; none/bc/abc proven byte-identical after it) and derives the config from the bc config.

JiT-S2-VMamba on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os

# 1) Pin torch to 2.5.1
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = "causal_conv1d-1.5.0.post8+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
MAMBA  = "mamba_ssm-2.2.4+cu12torch2.5cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The SSC vmamba.py (ssc: none|bc|abc), the build_model wiring
# in BOTH run_experiment.py and evaluate.py, the ssc configs, and the tests are
# all on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: confirm the ssc wiring is present in both entry points
!grep -q ssc scripts/run_experiment.py && grep -q ssc scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Add `ssc: "static"` support  *(idempotent patch; safe on resume)*

In [ ]:
# === Add ssc="static" (Mamba-3 bias-only control) to the cloned repo. ======
# Idempotent -- safe to re-run / leave in on resume. 5 anchored edits to
# src/models/vmamba.py (backup kept; syntax-checked; none/bc/abc proven
# byte-identical after the patch), then writes the ssc-static config.
# "static" = the bc arm with W_B/W_C removed entirely: only the ones-init
# per-direction biases B'=B+b0, C'=C+c0. Exactly what bc equals at step 0.
import py_compile, shutil, os

TARGET = "src/models/vmamba.py"
src = open(TARGET).read()

EDITS = [
    ('ssc: str = "none",            # "none" | "bc" | "abc"  (DiM-2 SSC)',
     'ssc: str = "none",            # "none" | "static" | "bc" | "abc"  (DiM-2 SSC; "static" = bias-only control)'),
    ('''        assert ssc in ("none", "bc", "abc"), ssc
        self.ssc = ssc
        if ssc != "none":
            self.bias_B = nn.Parameter(torch.ones(K, d_state))       # static, Mamba-3 style
            self.bias_C = nn.Parameter(torch.ones(K, d_state))
            self.cond_B_proj = nn.Linear(d_model, K * d_state, bias=False)  # zero-init (see initialize_weights)
            self.cond_C_proj = nn.Linear(d_model, K * d_state, bias=False)''',
     '''        assert ssc in ("none", "static", "bc", "abc"), ssc
        self.ssc = ssc
        if ssc != "none":
            self.bias_B = nn.Parameter(torch.ones(K, d_state))       # static, Mamba-3 style
            self.bias_C = nn.Parameter(torch.ones(K, d_state))
        if ssc in ("bc", "abc"):
            self.cond_B_proj = nn.Linear(d_model, K * d_state, bias=False)  # zero-init (see initialize_weights)
            self.cond_C_proj = nn.Linear(d_model, K * d_state, bias=False)'''),
    ('''        ssc_gate = None
        if self.ssc != "none" and cond is not None:
            assert extra_len == 0 and self.state_init == "none", (
                "ssc, state_init and the in-context prefix are separate "
                "conditioning arms; enable at most one per run."
            )
            B_ssm = B_ssm + (
                self.bias_B[None, :, :, None]
                + self.cond_B_proj(cond).view(B, K, d_state)[..., None]
            ).to(B_ssm.dtype)
            C_ssm = C_ssm + (
                self.bias_C[None, :, :, None]
                + self.cond_C_proj(cond).view(B, K, d_state)[..., None]
            ).to(C_ssm.dtype)
            if self.ssc == "abc":''',
     '''        ssc_gate = None
        if self.ssc != "none" and (self.ssc == "static" or cond is not None):
            assert extra_len == 0 and self.state_init == "none", (
                "ssc, state_init and the in-context prefix are separate "
                "conditioning arms; enable at most one per run."
            )
            add_B = self.bias_B[None, :, :, None]
            add_C = self.bias_C[None, :, :, None]
            if self.ssc in ("bc", "abc"):
                add_B = add_B + self.cond_B_proj(cond).view(B, K, d_state)[..., None]
                add_C = add_C + self.cond_C_proj(cond).view(B, K, d_state)[..., None]
            B_ssm = B_ssm + add_B.to(B_ssm.dtype)
            C_ssm = C_ssm + add_C.to(C_ssm.dtype)
            if self.ssc == "abc":'''),
    ('''        #   "none" \u2192 baseline / other arms (unchanged, byte-for-byte)
        #   "bc"   \u2192 B' = B + b0 + W_B c, C' = C + c0 + W_C c''',
     '''        #   "none"   \u2192 baseline / other arms (unchanged, byte-for-byte)
        #   "static" \u2192 B' = B + b0, C' = C + c0 ONLY (Mamba-3 bias control;
        #              equals "bc" at step 0, no condition path \u2014 isolates
        #              the static-bias effect from the conditioning effect)
        #   "bc"   \u2192 B' = B + b0 + W_B c, C' = C + c0 + W_C c'''),
    ('''        if self.ssc != "none":
            for block in self.blocks:
                nn.init.constant_(block.mixer.cond_B_proj.weight, 0)
                nn.init.constant_(block.mixer.cond_C_proj.weight, 0)''',
     '''        if self.ssc in ("bc", "abc"):
            for block in self.blocks:
                nn.init.constant_(block.mixer.cond_B_proj.weight, 0)
                nn.init.constant_(block.mixer.cond_C_proj.weight, 0)'''),
]

if '"static"' in src:
    print("vmamba.py already supports ssc='static' -- patch skipped.")
else:
    shutil.copy(TARGET, TARGET + ".bak-static")
    for old, new in EDITS:
        assert src.count(old) == 1, "anchor not found/unique: " + old[:90]
        src = src.replace(old, new)
    open(TARGET, "w").write(src)
    try:
        py_compile.compile(TARGET, doraise=True)
    except Exception:
        shutil.copy(TARGET + ".bak-static", TARGET)
        raise
    print("Patched", TARGET, "(5 edits, syntax OK; backup: .bak-static)")

CFG_STATIC = "configs/tiny_imagenet/jit-s2-vmamba-ssc-static.yaml"
if not os.path.exists(CFG_STATIC):
    base = open("configs/tiny_imagenet/jit-s2-vmamba-ssc-bc.yaml").read()
    # derive from the bc config: same recipe, only the arm + names change
    for old, new in [
        ("ssc: bc                           # none | bc | abc",
         "ssc: static                       # none | static | bc | abc"),
        ("name: jit-s2-vmamba-tinyin-ssc-bc",
         "name: jit-s2-vmamba-tinyin-ssc-static"),
        ("output_dir: experiments/tiny_imagenet/jit-s2-vmamba-ssc-bc",
         "output_dir: experiments/tiny_imagenet/jit-s2-vmamba-ssc-static"),
    ]:
        assert base.count(old) == 1, old
        base = base.replace(old, new)
    open(CFG_STATIC, "w").write(base)
    print("Wrote", CFG_STATIC, "(derived from the bc config: same recipe, arm only)")
else:
    print(CFG_STATIC, "already exists -- left untouched.")


## 4. Verify the arm  *(first session only; ~1 min)*

In [ ]:
# === Verify the SSC arm (~1 min; first session only -- comment out on resume).
# 1) equivalence proofs: gate identity exp((s*Delta)A)=exp(Delta*A*s) with B/s,
#    s=1 consistency, bias linearity -- machine epsilon vs selective_scan_ref
# 2) model guards: bit-identity of ssc=none / init semantics / grads / gate /
#    arm exclusivity
# PYTHONPATH=. is REQUIRED: without it "from src.models.vmamba import" fails
# and the tests' try/except misreports it as a missing mamba_ssm (known issue).
# If (1) FAILS after a Kaggle torch/CUDA bump, STOP -- do not train on an
# untrusted kernel build.
!PYTHONPATH=. python tests/test_ssc_equivalence.py
!PYTHONPATH=. python tests/test_ssc_model.py
# 3) static smoke test on the GPU: construct + one forward pass
!PYTHONPATH=. python -c "import torch; from src.models.vmamba import JiTVMamba; \
m = JiTVMamba(input_size=64, patch_size=8, hidden_size=384, depth=2, num_classes=200, ssc='static').cuda(); \
x = torch.randn(2, 3, 64, 64, device='cuda'); t = torch.rand(2, device='cuda'); \
y = torch.randint(0, 200, (2,), device='cuda'); \
print('ssc=static forward OK:', tuple(m(x, t, y).shape))"

## 5. Download Tiny-ImageNet  *(to /tmp; comment out if you attach a dataset)*

In [ ]:
# Download Tiny-ImageNet-200 to /tmp (EPHEMERAL: keeps your saved Version small --
# only checkpoints go to /kaggle/working). Re-downloads each session (~2-3 min).
# Faster alternative: attach a Kaggle tiny-imagenet dataset and set DATA_DIR to it,
# then comment this cell out. Requires Internet ON.
import os, zipfile, glob, shutil
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
SRC      = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    if not os.path.exists(ZIP):
        print("downloading tiny-imagenet-200 (~240MB)...")
        os.system(f"wget -q -O {ZIP} {SRC}")
    print("extracting...")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

print("train classes:", len(glob.glob(tr+"/*")),
      "| val classes:", len(glob.glob(val+"/*")), "| DATA_DIR:", DATA_DIR)

## 6. Settings  *(edits the cloned config so train & eval agree)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
SSC        = "static"    # "static" (bias-only control) | "bc" | "abc"  -- this notebook: static
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = f"configs/tiny_imagenet/jit-s2-vmamba-ssc-{SSC}.yaml"
OUT_DIR = "/kaggle/working/exp_ssc_static"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). in_context_len stays 0 and state_init stays none: the arms are
# mutually exclusive (assert in the model).
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s = re.sub(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "ssc": SSC, "in_context_len": 0, "state_init": "none"})

# ── Resume across 12h sessions ─────────────────────────────────────
# 1st run: RESUME_INPUT = None. Next session: Save Version, attach THIS notebook's
# previous output as an input, set RESUME_INPUT to ".../exp_ssc_static".
RESUME_INPUT = None
import os
RESUME_CKPT = None
for c in ([os.path.join(RESUME_INPUT, "checkpoint-last.pt")] if RESUME_INPUT else []) + \
         [os.path.join(OUT_DIR, "checkpoint-last.pt")]:
    if c and os.path.exists(c):
        RESUME_CKPT = c; break
print("config :", CONFIG, "| ssc:", SSC)
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR, "| resume:", RESUME_CKPT or "(fresh)")

## 7. Train  *(Save Version before 12h; set RESUME_INPUT next session)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + ssc now live in the (edited) config; we only override
# run-specific knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 8. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
# ── EVALUATE (FID + IS + complexity). Comment out this whole cell to skip. ──
# Real reference = Tiny-IN val (10k). Uses cfg_scale 2.5 / interval [0.1,1] / EMA 1,
# matching the training config. Set --n_samples 50000 for the official-style number.
#import os
#EVAL_CKPT = f"{OUT_DIR}/checkpoint-best.pt"
#if not os.path.exists(EVAL_CKPT):
#    EVAL_CKPT = f"{OUT_DIR}/checkpoint-last.pt"
#EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
#            f"--n_samples 10000 --batch_size 200 --ema 1 "
#            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0")
#print(EVAL_CMD, "\n" + "="*70)
#!{EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- Keep `EPOCHS`, `cfg_scale`, and `DATA_DIR` identical across all SSC-ladder notebooks (static / bc / abc) AND the baseline / in-context / state-init arms, so the only variable is the conditioning.
- Data lives in `/tmp` (not saved in the Version) to keep commits small; it re-downloads each session. To skip that, attach a Kaggle tiny-imagenet dataset and set `DATA_DIR` to it.
- Params: **+1.5K** vs baseline (2×K×d_state per block — just the biases). No condition path through SSC by construction; if this arm matches `bc`, the bc gain was the Mamba-3 bias, not the conditioning.
- FID protocol parity: same seed (42), Heun 50, cfg 2.5 / [0.1, 1.0], EMA 1, 10K samples, val reference — identical across arms; only then are the deltas attributable to the arm.